In [1]:
import pandas as pd

# Defino o ponto de entrada para os dados do VAB a preços correntes.
# Esta é a variável de controle macroeconômico mais relevante do modelo,
# permitindo capturar a dinâmica da atividade econômica municipal.
caminho_arquivo = r'C:\Users\fabio\TCC\4_Pib_Munic.csv'

try:
    # Leitura do dataset do PIB Municipal (Série Histórica)
    # sep=';': Padrão de exportação de CSVs do IBGE/SIDRA.
    # skiprows=1: Eliminação de cabeçalhos institucionais não estruturados.
    df_pib = pd.read_csv(
        caminho_arquivo,
        skiprows=1,
        sep=';',
        encoding='utf-8-sig'
    )
    
    # DIAGNÓSTICO ESTRUTURAL INICIAL:
    # Verificação da integridade da carga e mapeamento das colunas temporais.
    print("--- Visualização do Dataset de Atividade Econômica (PIB) ---")
    print(df_pib.head())
    
    print("\n--- Inventário de Variáveis e Séries Temporais ---")
    print(df_pib.columns.tolist())

except Exception as e:
    print(f"ERRO DE INGESTÃO: Falha ao carregar a matriz do PIB: {e}")

--- Visualização do Dataset de Atividade Econômica (PIB) ---
  Sigla   Código     Município         1985         1996         1999  \
0    AC  1200013    Acrelândia            0   21208,6573  40965,13598   
1    AC  1200054  Assis Brasil  6371,510267   13109,9068  17248,27103   
2    AC  1200104     Brasiléia   106133,915  92066,11406  124153,2916   
3    AC  1200138        Bujari            0  35620,65165  45273,76357   
4    AC  1200179      Capixaba            0  25127,46237  30424,09048   

          2000         2001         2002         2003  ...         2012  \
0  50440,13137  72381,18354  75272,10182  81950,12668  ...  138723,5425   
1  20008,00807  20493,96641  24295,16876  24334,85293  ...  45162,40336   
2  135485,4795  116439,6071   138675,481  143296,3802  ...   204155,787   
3  44720,44638  53672,45595   50804,0011  59570,65397  ...  92295,34316   
4  36839,95905  62355,87119  61727,18792  65467,87067  ...  110884,5835   

          2013         2014         2015         

In [2]:
# --- ETAPA 2: REESTRUTURAÇÃO PARA PAINEL DE DADOS (LONG FORMAT) ---

# Para que o modelo econométrico (Controle Sintético / Diferença em Diferenças) 
# processe a série histórica do VAB, converto a matriz horizontal (Wide) 
# em um formato vertical (Long). Este procedimento é essencial para a 
# compatibilidade com o comando 'xtset' do Stata.

print("Iniciando o Reshaping da matriz de PIB...")

# 1. DEFINIÇÃO DAS CHAVES DE IDENTIFICAÇÃO (ID_VARS)
# Mantenho as dimensões geográficas fixas para cada registro anual.
eixos_identidade = ['Sigla', 'Código', 'Município']

# 2. OPERAÇÃO DE 'MELT' (TRANSPOSIÇÃO)
# Converto os atributos de anos em uma variável temporal ('Ano') 
# e o montante financeiro em uma variável métrica ('PIB').
df_pib_longo = df_pib.melt(
    id_vars=eixos_identidade,
    var_name='Ano',
    value_name='PIB'
)

# 3. AUDITORIA DE ESTRUTURA E CRONOLOGIA
# Verifico a integridade das primeiras e últimas observações para 
# validar se o empilhamento dos anos ocorreu sem fragmentação de dados.
print("\n--- Diagnóstico: Painel de Atividade Econômica (PIB) ---")
print("Cabeçalho (Início da Série):")
print(df_pib_longo.head())

print("\nRodapé (Fim da Série):")
print(df_pib_longo.tail())

Iniciando o Reshaping da matriz de PIB...

--- Diagnóstico: Painel de Atividade Econômica (PIB) ---
Cabeçalho (Início da Série):
  Sigla   Código     Município   Ano          PIB
0    AC  1200013    Acrelândia  1985            0
1    AC  1200054  Assis Brasil  1985  6371,510267
2    AC  1200104     Brasiléia  1985   106133,915
3    AC  1200138        Bujari  1985            0
4    AC  1200179      Capixaba  1985            0

Rodapé (Fim da Série):
       Sigla   Código       Município   Ano          PIB
139920    TO  1721208  Tocantinópolis  2021  173952,0019
139921    TO  1721257        Tupirama  2021  55474,91145
139922    TO  1721307      Tupiratins  2021   18027,6307
139923    TO  1722081    Wanderlândia  2021  85623,66022
139924    TO  1722107         Xambioá  2021  219875,5694


In [3]:
# --- ETAPA 3: SERIALIZAÇÃO E PERSISTÊNCIA DA MATRIZ (OUTPUT) ---

# Concluo o processamento da variável de atividade econômica exportando o dataset 
# consolidado em formato 'Long'. Este arquivo é o pilar central para a construção 
# do cenário contrafactual e para as estimações de crescimento econômico do TCC.

# 1. DEFINIÇÃO DO DIRETÓRIO DE SAÍDA
# Armazeno o dataset final utilizando nomenclatura padronizada para assegurar 
# a organização do pipeline de dados no diretório do projeto.
caminho_saida_csv = r'C:\Users\fabio\TCC\VARIAVEIS\4_VAB_FINAL.csv'

# 2. EXPORTAÇÃO E CONFORMIDADE ESTRUTURAL
# Utilizo o parâmetro index=False para garantir uma estrutura de dados limpa, 
# facilitando a importação direta para softwares estatísticos (Stata) 
# sem a necessidade de novos tratamentos de colunas residuais.
df_pib_longo.to_csv(
    caminho_saida_csv,
    index=False
)

print(f"--- PROTOCOLO DE EXPORTAÇÃO CONCLUÍDO ---")
print(f"Dataset de PIB Municipal Consolidado: {caminho_saida_csv}")

--- PROTOCOLO DE EXPORTAÇÃO CONCLUÍDO ---
Dataset de PIB Municipal Consolidado: C:\Users\fabio\TCC\VARIAVEIS\4_VAB_FINAL.csv
